 FIFA WORLD CUP 2026 — NETWORK ANALYSIS USING GRAPH THEORY.               
    Author   : Mariam Adejoke Popoola (PhD Student in Informatics )— UFAM, Manaus, Brazil.             ║
   
  Libraries: pandas, numpy, networkx, matplotlib, seaborn, scikit-learn.    ║


ACADEMIC BACKGROUND
───────────────────
Graph Theory (Euler, 1736) provides the mathematical foundation for this
project.  A graph G = (V, E) consists of a finite set of vertices V and a
set of edges E ⊆ V × V.

Mapping to the World Cup:
  • V  →  Teams (48 nations competing in 2026)
  • E  →  Matches (72 confirmed group-stage fixtures)

Two complementary graph models are built:

  G_match  (undirected)
      Edge exists when two teams play each other.
      Group Stage only → 12 disjoint complete sub-graphs.

  G_city   (undirected, weighted)
      Edge exists when two DIFFERENT teams play in the SAME host city.
      Weight = number of cities shared.
      Enables meaningful centrality variation across the full 48-team set.

All nine project phases are implemented below with extensive comments
written for a graduate Computer Science audience.


In [1]:
# Cell 1: Importation of Libraries
import os
import warnings
from collections import deque

import matplotlib
matplotlib.use("Agg")                      # headless / server-safe backend

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import numpy  as np
import pandas as pd
import networkx as nx
import seaborn as sns

warnings.filterwarnings("ignore")


In [2]:
BASE  = "/content/"
DATA  = os.path.join(BASE, "data")
FIGS  = os.path.join(BASE, "outputs", "figures")
RPTS  = os.path.join(BASE, "outputs", "reports")
os.makedirs(FIGS, exist_ok=True)
os.makedirs(RPTS, exist_ok=True)

# ──────────────────────────────────────────────────────────────────────────────
#  COLOUR PALETTE  (one consistent colour per group A–L throughout ALL figures)
# ──────────────────────────────────────────────────────────────────────────────
GROUPS      = list("ABCDEFGHIJKL")
_cmap       = cm.get_cmap("tab20", 12)
GROUP_COLOR = {g: _cmap(i) for i, g in enumerate(GROUPS)}


In [3]:
# Cell 2 — DATA UNDERSTANDING  ░░
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 70)
print("  PHASE 1 — DATA UNDERSTANDING")
print("═" * 70)

# ── 1·1  Load datasets ───────────────────────────────────────────────────────
#
# pandas.read_csv() parses CSV files into DataFrame objects — the core
# two-dimensional data structure in the pandas ecosystem.
# Each DataFrame stores typed columns (int64, float64, object, bool).
#
matches = pd.read_csv(os.path.join(DATA, "/content/matches.csv"))
teams   = pd.read_csv(os.path.join(DATA, "/content/teams.csv"))
cities  = pd.read_csv(os.path.join(DATA, "/content/host_cities.csv"))
stages  = pd.read_csv(os.path.join(DATA, "/content/tournament_stages.csv"))

DATASETS = {
    "matches"            : matches,
    "teams"              : teams,
    "host_cities"        : cities,
    "tournament_stages"  : stages,
}



══════════════════════════════════════════════════════════════════════
  PHASE 1 — DATA UNDERSTANDING
══════════════════════════════════════════════════════════════════════


In [4]:
# ── 1·2  Dataset dimensions ──────────────────────────────────────────────────
print("\n┌─ 1·2  Dataset Dimensions ──────────────────────────────────────────┐")
for name, df in DATASETS.items():
    r, c = df.shape
    print(f"│  {name:<22}  →  {r:>4} rows  ×  {c} columns")
print("└────────────────────────────────────────────────────────────────────┘")



┌─ 1·2  Dataset Dimensions ──────────────────────────────────────────┐
│  matches                 →   104 rows  ×  8 columns
│  teams                   →    48 rows  ×  5 columns
│  host_cities             →    16 rows  ×  6 columns
│  tournament_stages       →     7 rows  ×  3 columns
└────────────────────────────────────────────────────────────────────┘


In [5]:
# ── 1·3  Column information (dtype + null counts) ────────────────────────────
print("\n┌─ 1·3  Column Information ───────────────────────────────────────────┐")
for name, df in DATASETS.items():
    info = pd.DataFrame({
        "dtype"   : df.dtypes.astype(str),
        "non_null": df.notnull().sum(),
        "null"    : df.isnull().sum(),
    })
    print(f"\n  [{name}]")
    print(info.to_string())
print()



┌─ 1·3  Column Information ───────────────────────────────────────────┐

  [matches]
                dtype  non_null  null
id              int64       104     0
match_number    int64       104     0
home_team_id  float64        72    32
away_team_id  float64        72    32
city_id         int64       104     0
stage_id        int64       104     0
kickoff_at     object       104     0
match_label    object       104     0

  [teams]
                 dtype  non_null  null
id               int64        48     0
team_name       object        48     0
fifa_code       object        48     0
group_letter    object        48     0
is_placeholder    bool        48     0

  [host_cities]
                 dtype  non_null  null
id               int64        16     0
city_name       object        16     0
country         object        16     0
venue_name      object        16     0
region_cluster  object        16     0
airport_code    object        16     0

  [tournament_stages]
              

In [6]:
# ── 1·4  Missing values ──────────────────────────────────────────────────────
print("┌─ 1·4  Missing Values ───────────────────────────────────────────────┐")
for name, df in DATASETS.items():
    total = df.isnull().sum().sum()
    flag  = "⚠" if total > 0 else "✓"
    print(f"│  {flag}  {name:<22}  {total} missing cell(s)")
    if total > 0:
        detail = df.isnull().sum()[df.isnull().sum() > 0]
        for col, cnt in detail.items():
            print(f"│       └─ {col}: {cnt} nulls")
print("│")
print("│  INTERPRETATION:")
print("│  matches.home_team_id and matches.away_team_id each carry 32 nulls.")
print("│  These correspond exactly to the 32 knockout fixtures (Round of 32")
print("│  through the Final) where participants are unknown until the Group")
print("│  Stage concludes. This is NOT a data-quality problem — it is an")
print("│  intrinsic property of a tournament bracket.")
print("└────────────────────────────────────────────────────────────────────┘")


┌─ 1·4  Missing Values ───────────────────────────────────────────────┐
│  ⚠  matches                 64 missing cell(s)
│       └─ home_team_id: 32 nulls
│       └─ away_team_id: 32 nulls
│  ✓  teams                   0 missing cell(s)
│  ✓  host_cities             0 missing cell(s)
│  ✓  tournament_stages       0 missing cell(s)
│
│  INTERPRETATION:
│  matches.home_team_id and matches.away_team_id each carry 32 nulls.
│  These correspond exactly to the 32 knockout fixtures (Round of 32
│  through the Final) where participants are unknown until the Group
│  Stage concludes. This is NOT a data-quality problem — it is an
│  intrinsic property of a tournament bracket.
└────────────────────────────────────────────────────────────────────┘


In [7]:
# ── 1·5  Duplicate records ───────────────────────────────────────────────────
print("\n┌─ 1·5  Duplicate Records ────────────────────────────────────────────┐")
for name, df in DATASETS.items():
    dupes = df.duplicated().sum()
    flag  = "⚠" if dupes > 0 else "✓"
    print(f"│  {flag}  {name:<22}  {dupes} duplicate row(s)")
print("└────────────────────────────────────────────────────────────────────┘")



┌─ 1·5  Duplicate Records ────────────────────────────────────────────┐
│  ✓  matches                 0 duplicate row(s)
│  ✓  teams                   0 duplicate row(s)
│  ✓  host_cities             0 duplicate row(s)
│  ✓  tournament_stages       0 duplicate row(s)
└────────────────────────────────────────────────────────────────────┘


In [8]:
# ── 1·6  Descriptive statistics & EDA ───────────────────────────────────────
print("\n┌─ 1·6  Descriptive Statistics & EDA ────────────────────────────────┐")

print("\n  [matches] — numeric summary:")
print(matches.describe().round(2).to_string())

print("\n  [teams] — real teams per group:")
real_teams = teams[~teams["is_placeholder"]].copy()
grp_counts = real_teams.groupby("group_letter").size().rename("n_real_teams")
print(grp_counts.to_string())

print("\n  [teams] — placeholder count:")
print(f"  {teams['is_placeholder'].sum()} placeholder team(s) (e.g. 'Winner UEFA Playoff D')")

print("\n  [host_cities] — venues by region cluster:")
print(cities.groupby("region_cluster").size().rename("n_cities").to_string())

print("\n  [host_cities] — venues by country:")
print(cities.groupby("country").size().rename("n_venues").to_string())

print("\n  [tournament_stages] — all 7 stages:")
print(stages.to_string(index=False))

print("\n  [matches] — matches per stage:")
stage_counts = (matches
                .merge(stages, left_on="stage_id", right_on="id")
                ["stage_name"].value_counts()
                .sort_index()
                .rename("n_matches"))
print(stage_counts.to_string())

print("\n  [matches] — kickoff date range:")
matches["kickoff_dt"] = pd.to_datetime(matches["kickoff_at"], utc=True)
print(f"  First match : {matches['kickoff_dt'].min().date()}")
print(f"  Last  match : {matches['kickoff_dt'].max().date()}")
print(f"  Tournament span: {(matches['kickoff_dt'].max()-matches['kickoff_dt'].min()).days} days")



┌─ 1·6  Descriptive Statistics & EDA ────────────────────────────────┐

  [matches] — numeric summary:
           id  match_number  home_team_id  away_team_id  city_id  stage_id
count  104.00        104.00         72.00         72.00   104.00    104.00
mean    52.50         52.50         24.44         24.56     7.69      1.61
std     30.17         30.17         13.92         13.99     4.52      1.16
min      1.00          1.00          1.00          1.00     1.00      1.00
25%     26.75         26.75         12.75         12.75     4.00      1.00
50%     52.50         52.50         24.50         24.50     7.50      1.00
75%     78.25         78.25         36.25         36.25    11.25      2.00
max    104.00        104.00         48.00         48.00    16.00      7.00

  [teams] — real teams per group:
group_letter
A    3
B    3
C    4
D    3
E    4
F    3
G    4
H    4
I    3
J    4
K    3
L    4

  [teams] — placeholder count:
  6 placeholder team(s) (e.g. 'Winner UEFA Playoff D')

 

In [9]:
# ── 1·7  Purpose of each dataset ─────────────────────────────────────────────
print("\n┌─ 1·7  Dataset Purposes ─────────────────────────────────────────────┐")
purposes = [
    ("matches.csv",
     "EDGE TABLE. Each row is one scheduled match (an edge in G_match)\n"
     "│    connecting home_team_id and away_team_id (vertices). Also\n"
     "│    links to host city and tournament stage for edge attributes."),
    ("teams.csv",
     "VERTEX ATTRIBUTE TABLE. Provides country name, FIFA 3-letter\n"
     "│    code, group assignment A-L, and placeholder flag for each\n"
     "│    of the 48 participating nations."),
    ("host_cities.csv",
     "GEOGRAPHIC CONTEXT. Links each match to its host city, country\n"
     "│    (USA / Canada / Mexico), venue name, region cluster, and\n"
     "│    IATA airport code."),
    ("tournament_stages.csv",
     "EDGE CLASSIFICATION TABLE. Names and orders the 7 tournament\n"
     "│    stages from Group Stage (1) through the Final (7)."),
]
for fname, desc in purposes:
    print(f"│\n│  {fname}\n│    {desc}")
print("└────────────────────────────────────────────────────────────────────┘")



┌─ 1·7  Dataset Purposes ─────────────────────────────────────────────┐
│
│  matches.csv
│    EDGE TABLE. Each row is one scheduled match (an edge in G_match)
│    connecting home_team_id and away_team_id (vertices). Also
│    links to host city and tournament stage for edge attributes.
│
│  teams.csv
│    VERTEX ATTRIBUTE TABLE. Provides country name, FIFA 3-letter
│    code, group assignment A-L, and placeholder flag for each
│    of the 48 participating nations.
│
│  host_cities.csv
│    GEOGRAPHIC CONTEXT. Links each match to its host city, country
│    (USA / Canada / Mexico), venue name, region cluster, and
│    IATA airport code.
│
│  tournament_stages.csv
│    EDGE CLASSIFICATION TABLE. Names and orders the 7 tournament
│    stages from Group Stage (1) through the Final (7).
└────────────────────────────────────────────────────────────────────┘


In [10]:
# ░░  PHASE 2 — DATA PREPROCESSING  ░░
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 70)
print("  PHASE 2 — DATA PREPROCESSING")
print("═" * 70)



══════════════════════════════════════════════════════════════════════
  PHASE 2 — DATA PREPROCESSING
══════════════════════════════════════════════════════════════════════


In [11]:
# ░░  PHASE 2 — DATA PREPROCESSING  ░░
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 70)
print("  PHASE 2 — DATA PREPROCESSING")
print("═" * 70)

# ── Step 2·1  Build an ID → team_name lookup dictionary ──────────────────────
#
# We use a Python dict for O(1) lookups rather than repeated DataFrame merges.
# The dict maps integer team IDs to their string country names.
#
id_to_name  = dict(zip(teams["id"], teams["team_name"]))
id_to_group = dict(zip(teams["id"], teams["group_letter"]))
id_to_code  = dict(zip(teams["id"], teams["fifa_code"]))
print("\n  Step 2·1 — id→name lookup built  ({} entries)".format(len(id_to_name)))



══════════════════════════════════════════════════════════════════════
  PHASE 2 — DATA PREPROCESSING
══════════════════════════════════════════════════════════════════════

  Step 2·1 — id→name lookup built  (48 entries)


In [20]:
# ── Step 2·2  Map numeric IDs to country names in the matches table ──────────
#
# pandas .map() applies the dictionary element-wise over a Series.
# NaN IDs (knockout fixtures) remain NaN after mapping.
#
clean = matches.copy()
clean["home_team"]  = clean["home_team_id"].map(id_to_name)
clean["away_team"]  = clean["away_team_id"].map(id_to_name)
clean["home_group"] = clean["home_team_id"].map(id_to_group)
print("  Step 2·2 — team IDs replaced with country names")


  Step 2·2 — team IDs replaced with country names


In [21]:
# ── Step 2·3  Merge stage names and city names ───────────────────────────────
clean = (clean
         .merge(stages[["id","stage_name"]], left_on="stage_id", right_on="id",
                suffixes=("","_stage"))
         .merge(cities[["id","city_name","country","region_cluster","venue_name"]],
                left_on="city_id", right_on="id",
                suffixes=("","_city")))
clean.rename(columns={"stage_name": "match_stage"}, inplace=True)
print("  Step 2·3 — stage names and city names merged")


  Step 2·3 — stage names and city names merged


In [22]:
# ── Step 2·4  Parse timestamps to UTC-aware datetime ─────────────────────────
clean["match_date"] = pd.to_datetime(clean["kickoff_at"], utc=True)
print("  Step 2·4 — kickoff_at parsed to UTC datetime64")


  Step 2·4 — kickoff_at parsed to UTC datetime64


In [23]:
# ── Step 2·5  Select final clean columns ─────────────────────────────────────
CLEAN_COLS = ["match_number","home_team","away_team","match_stage",
              "match_date","city_name","country","region_cluster",
              "venue_name","match_label"]
clean_df   = clean[CLEAN_COLS].copy()
print("  Step 2·5 — final clean dataset assembled")

print("\n  ── Preview (first 12 group-stage rows) ─────────────────────────────")
gs_preview = clean_df[clean_df["match_stage"]=="Group Stage"].head(12)
print(gs_preview[["match_number","home_team","away_team","match_stage",
                   "city_name","country"]].to_string(index=False))


  Step 2·5 — final clean dataset assembled

  ── Preview (first 12 group-stage rows) ─────────────────────────────
 match_number     home_team             away_team match_stage              city_name country
            1        Mexico          South Africa Group Stage            Mexico City  Mexico
            2   South Korea Winner UEFA Playoff D Group Stage            Guadalajara  Mexico
            3        Canada Winner UEFA Playoff A Group Stage                Toronto  Canada
            4           USA              Paraguay Group Stage            Los Angeles     USA
            5         Haiti              Scotland Group Stage                 Boston     USA
            6        Brazil               Morocco Group Stage    New York/New Jersey     USA
            7         Qatar           Switzerland Group Stage San Francisco Bay Area     USA
            8     Australia Winner UEFA Playoff C Group Stage              Vancouver  Canada
            9       Germany               Curaça

In [24]:
# ── Step 2·6  Isolate Group Stage matches with confirmed teams ────────────────
gs_df = clean_df[
    (clean_df["match_stage"] == "Group Stage") &
    clean_df["home_team"].notna() &
    clean_df["away_team"].notna()
].copy()

print(f"\n  Total matches          : {len(clean_df)}")
print(f"  Confirmed Group Stage  : {len(gs_df)}")
print(f"  Unresolved (knockout)  : {clean_df['home_team'].isna().sum()}")



  Total matches          : 104
  Confirmed Group Stage  : 72
  Unresolved (knockout)  : 32


In [25]:
# ── Step 2·7  Build per-team city lookup (used in G_city later) ──────────────
#
# For each team, collect every host city where they play a group match.
# This will let us connect teams that share a venue in Phase 3.
#
team_cities: dict[str, set] = {}
for _, row in gs_df.iterrows():
    for team in (row["home_team"], row["away_team"]):
        if pd.notna(team):
            team_cities.setdefault(team, set()).add(row["city_name"])
print(f"\n  Step 2·7 — city sets built for {len(team_cities)} teams")



  Step 2·7 — city sets built for 48 teams


In [26]:
# ── Step 2·8  Export clean dataset ───────────────────────────────────────────
clean_path = os.path.join(RPTS, "clean_matches.csv")
clean_df.to_csv(clean_path, index=False)
print(f"  Step 2·8 — clean_matches.csv saved → {clean_path}")



  Step 2·8 — clean_matches.csv saved → /content/outputs/reports/clean_matches.csv


In [27]:
# ░░  PHASE 3 — GRAPH CONSTRUCTION  ░░
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 70)
print("  PHASE 3 — GRAPH CONSTRUCTION")
print("═" * 70)

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  GRAPH 1 — G_match  (the match graph)                                   │
# │  G_match = (V, E)                                                        │
# │  V = real (non-placeholder) teams  |V| ≤ 48                             │
# │  E = {(h,a) | match exists between home h and away a, Group Stage only} │
# │                                                                          │
# │  Mathematical note:                                                      │
# │  The Group Stage is a round-robin within each group of 3 or 4 teams.   │
# │  A round-robin on n teams forms K_n (the complete graph on n vertices). │
# │  With 12 groups of 4: 12 × K_4.  K_4 has C(4,2) = 6 edges.           │
# │  Groups with 3 teams form K_3 with C(3,2) = 3 edges.                  │
# │  Crucially, K_4 is a REGULAR graph: every vertex has degree 3.         │
# │  This means ALL degree-based centralities will be UNIFORM on G_match.  │
# └─────────────────────────────────────────────────────────────────────────┘

G_match = nx.Graph(name="FIFA WC 2026 – Match Graph (Group Stage)")

# Add nodes with group and FIFA-code attributes
for _, row in real_teams.iterrows():
    G_match.add_node(row["team_name"],
                     group     = row["group_letter"],
                     fifa_code = row["fifa_code"])

# Add edges (one edge per match)
for _, row in gs_df.iterrows():
    h, a = row["home_team"], row["away_team"]
    # Only add edge if both endpoints are real (non-placeholder) teams
    if h in G_match.nodes and a in G_match.nodes:
        G_match.add_edge(h, a,
                         city  = row["city_name"],
                         stage = row["match_stage"],
                         date  = str(row["match_date"].date()))

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  GRAPH 2 — G_city  (city-sharing graph, the MAIN ANALYTICAL GRAPH)      │
# │                                                                          │
# │  Motivation: G_match yields UNIFORM centrality (every node has degree   │
# │  = 3) because the tournament design enforces equality. To perform        │
# │  meaningful centrality analysis we need a graph where connectivity       │
# │  varies.  G_city achieves this by connecting teams through SHARED HOST  │
# │  CITIES rather than direct matches.                                      │
# │                                                                          │
# │  V = all real teams (same as G_match)                                   │
# │  E = {(t1,t2) | t1 ≠ t2 ∧ ∃ city c : t1 plays in c ∧ t2 plays in c} │
# │  weight(t1,t2) = |shared cities between t1 and t2|                     │
# └─────────────────────────────────────────────────────────────────────────┘

G_city = nx.Graph(name="FIFA WC 2026 – City-Sharing Graph")

# Copy all nodes from G_match (same vertex set)
for node, attrs in G_match.nodes(data=True):
    G_city.add_node(node, **attrs)

# Add edges for every pair sharing at least one host city
node_list = list(G_city.nodes())
for i in range(len(node_list)):
    for j in range(i + 1, len(node_list)):
        t1, t2    = node_list[i], node_list[j]
        shared    = team_cities.get(t1, set()) & team_cities.get(t2, set())
        if shared:
            G_city.add_edge(t1, t2,
                            shared_cities = sorted(shared),
                            weight        = len(shared))



══════════════════════════════════════════════════════════════════════
  PHASE 3 — GRAPH CONSTRUCTION
══════════════════════════════════════════════════════════════════════


In [28]:
# ── 3·1  G_match metrics ─────────────────────────────────────────────────────
print("\n┌─ 3·1  G_match — Match Graph Metrics ───────────────────────────────┐")
n_nodes  = G_match.number_of_nodes()
n_edges  = G_match.number_of_edges()
density  = nx.density(G_match)
comps    = list(nx.connected_components(G_match))
n_comps  = len(comps)
degrees  = [d for _, d in G_match.degree()]

print(f"│  Vertices (teams)          : {n_nodes}")
print(f"│  Edges (matches)           : {n_edges}")
print(f"│  Graph density             : {density:.6f}")
print(f"│  Connected components      : {n_comps}")
print(f"│  Avg degree                : {np.mean(degrees):.2f} (expected = 3.0 for K_4 groups)")
print(f"│")
print(f"│  EXPLANATION OF METRICS:")
print(f"│  • Density = |E| / [|V|×(|V|−1)/2] = {n_edges}/{n_nodes*(n_nodes-1)//2} = {density:.6f}")
print(f"│    Density → 0 means sparse (few edges vs. possible pairs).")
print(f"│  • 12 connected components confirm the structure: one component")
print(f"│    per group (A–L). Teams only connect to others in their group.")
print(f"│  • Uniform degree = 3: every team plays exactly 3 group matches,")
print(f"│    making this a 3-regular graph (each K_4 is 3-regular).")
print(f"└────────────────────────────────────────────────────────────────────┘")

print("\n  Component breakdown (groups):")
for cc in sorted(comps, key=lambda s: next(iter(s))):
    sample = sorted(cc)[0]
    grp    = G_match.nodes[sample].get("group","?")
    print(f"    Group {grp}: {sorted(cc)}")

# ── 3·2  G_city metrics ──────────────────────────────────────────────────────
print("\n┌─ 3·2  G_city — City-Sharing Graph Metrics ─────────────────────────┐")
cn = G_city.number_of_nodes()
ce = G_city.number_of_edges()
cd = nx.density(G_city)
cc2 = nx.number_connected_components(G_city)
print(f"│  Vertices                  : {cn}")
print(f"│  Edges                     : {ce}")
print(f"│  Density                   : {cd:.4f}")
print(f"│  Connected components      : {cc2}")
print(f"│")
print(f"│  EXPLANATION:")
print(f"│  G_city is far denser (density ≈ {cd:.2f}) because host cities serve")
print(f"│  as hubs connecting teams from multiple groups. Teams playing in")
print(f"│  New York, Dallas, or Los Angeles share those cities with teams")
print(f"│  from entirely different groups, creating rich cross-group edges.")
print(f"└────────────────────────────────────────────────────────────────────┘")



┌─ 3·1  G_match — Match Graph Metrics ───────────────────────────────┐
│  Vertices (teams)          : 42
│  Edges (matches)           : 54
│  Graph density             : 0.062718
│  Connected components      : 12
│  Avg degree                : 2.57 (expected = 3.0 for K_4 groups)
│
│  EXPLANATION OF METRICS:
│  • Density = |E| / [|V|×(|V|−1)/2] = 54/861 = 0.062718
│    Density → 0 means sparse (few edges vs. possible pairs).
│  • 12 connected components confirm the structure: one component
│    per group (A–L). Teams only connect to others in their group.
│  • Uniform degree = 3: every team plays exactly 3 group matches,
│    making this a 3-regular graph (each K_4 is 3-regular).
└────────────────────────────────────────────────────────────────────┘

  Component breakdown (groups):
    Group J: ['Algeria', 'Argentina', 'Austria', 'Jordan']
    Group G: ['Belgium', 'Egypt', 'IR Iran', 'New Zealand']
    Group E: ['Curaçao', "Côte d'Ivoire", 'Ecuador', 'Germany']
    Group C: ['Brazil',

In [29]:
# ░░  PHASE 4 — NETWORK VISUALIZATION  ░░
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 70)
print("  PHASE 4 — NETWORK VISUALIZATION  (saving 7 PNG figures)")
print("═" * 70)

def node_color_list(G):
    """Return a list of RGBA colours, one per node, based on group letter."""
    return [GROUP_COLOR.get(G.nodes[n].get("group","?"), "#888888")
            for n in G.nodes()]

def group_legend(ax, ncol=2):
    """Add a consistent group-colour legend to an axes."""
    patches = [mpatches.Patch(color=GROUP_COLOR[g], label=f"Group {g}")
               for g in GROUPS]
    ax.legend(handles=patches, loc="upper left", ncol=ncol,
              fontsize=8, title="Groups", title_fontsize=9,
              framealpha=0.85)
# ── Fig 1 — Full Group Stage Match Network ────────────────────────────────────
print("\n  → Fig 1: Full Group Stage Match Network")
fig, ax = plt.subplots(figsize=(22, 16))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#0d1117")

pos = nx.spring_layout(G_match, seed=42, k=2.5)

nx.draw_networkx_edges(G_match, pos, ax=ax,
                       alpha=0.45, edge_color="#aaaaaa", width=1.4)
nx.draw_networkx_nodes(G_match, pos, ax=ax,
                       node_color=node_color_list(G_match),
                       node_size=1000, linewidths=1.5, edgecolors="white")
nx.draw_networkx_labels(G_match, pos, ax=ax,
                        font_size=6.5, font_color="white", font_weight="bold")

group_legend(ax, ncol=2)
ax.set_title("FIFA World Cup 2026 — Full Group Stage Network\n"
             "Nodes = Teams  │  Edges = Scheduled Matches  │  Colours = Group Assignment",
             color="white", fontsize=14, fontweight="bold", pad=18)
ax.axis("off")
plt.tight_layout()
p = os.path.join(FIGS, "fig1_full_match_network.png")
plt.savefig(p, dpi=160, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"    ✓ {p}")


══════════════════════════════════════════════════════════════════════
  PHASE 4 — NETWORK VISUALIZATION  (saving 7 PNG figures)
══════════════════════════════════════════════════════════════════════

  → Fig 1: Full Group Stage Match Network
    ✓ /content/outputs/figures/fig1_full_match_network.png


In [30]:
# ── Fig 1 — Full Group Stage Match Network ────────────────────────────────────
print("\n  → Fig 1: Full Group Stage Match Network")
fig, ax = plt.subplots(figsize=(22, 16))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#0d1117")

pos = nx.spring_layout(G_match, seed=42, k=2.5)

nx.draw_networkx_edges(G_match, pos, ax=ax,
                       alpha=0.45, edge_color="#aaaaaa", width=1.4)
nx.draw_networkx_nodes(G_match, pos, ax=ax,
                       node_color=node_color_list(G_match),
                       node_size=1000, linewidths=1.5, edgecolors="white")
nx.draw_networkx_labels(G_match, pos, ax=ax,
                        font_size=6.5, font_color="white", font_weight="bold")

group_legend(ax, ncol=2)
ax.set_title("FIFA World Cup 2026 — Full Group Stage Network\n"
             "Nodes = Teams  │  Edges = Scheduled Matches  │  Colours = Group Assignment",
             color="white", fontsize=14, fontweight="bold", pad=18)
ax.axis("off")
plt.tight_layout()
p = os.path.join(FIGS, "fig1_full_match_network.png")
plt.savefig(p, dpi=160, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"    ✓ {p}")



  → Fig 1: Full Group Stage Match Network
    ✓ /content/outputs/figures/fig1_full_match_network.png


In [31]:
# ── Fig 2 — Group Sub-Networks (3 × 4 grid) ───────────────────────────────────
print("  → Fig 2: Group-level networks (3×4 grid)")
fig, axes = plt.subplots(3, 4, figsize=(24, 18))
fig.patch.set_facecolor("#0d1117")
axes_flat = axes.flatten()

for i, grp in enumerate(GROUPS):
    ax  = axes_flat[i]
    ax.set_facecolor("#111827")
    members = [n for n, d in G_match.nodes(data=True) if d.get("group") == grp]
    H       = G_match.subgraph(members)
    pos_h   = nx.circular_layout(H)
    col     = GROUP_COLOR[grp]

    nx.draw_networkx_edges(H, pos_h, ax=ax,
                           alpha=0.7, edge_color="#cccccc", width=2.2)
    nx.draw_networkx_nodes(H, pos_h, ax=ax,
                           node_color=[col]*len(H),
                           node_size=1600, linewidths=2, edgecolors="white")
    nx.draw_networkx_labels(H, pos_h, ax=ax,
                            font_size=7, font_color="white", font_weight="bold")

    # FIFA codes below node labels
    codes = {n: G_match.nodes[n].get("fifa_code","") for n in H.nodes()}
    pos_lo = {k: (v[0], v[1] - 0.28) for k, v in pos_h.items()}
    nx.draw_networkx_labels(H, pos_lo, labels=codes, ax=ax,
                            font_size=7.5, font_color="#aaaaff")

    ax.set_title(f"Group {grp}  ({H.number_of_edges()} matches  ·  {H.number_of_nodes()} teams)",
                 fontsize=11, fontweight="bold", color=col, pad=8)
    ax.axis("off")

fig.suptitle("FIFA World Cup 2026 — Group-Level Networks  (K₄ / K₃ Cliques)",
             fontsize=16, fontweight="bold", color="white", y=1.01)
plt.tight_layout()
p = os.path.join(FIGS, "fig2_group_networks.png")
plt.savefig(p, dpi=160, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"    ✓ {p}")


  → Fig 2: Group-level networks (3×4 grid)
    ✓ /content/outputs/figures/fig2_group_networks.png


In [32]:
# ── Fig 3 — Degree Distribution ───────────────────────────────────────────────
print("  → Fig 3: Degree distribution")
fig, ax = plt.subplots(figsize=(9, 5))
degs = [d for _, d in G_match.degree()]
unique_d = sorted(set(degs))
counts   = [degs.count(d) for d in unique_d]
bars = ax.bar(unique_d, counts, color="#2563eb", edgecolor="white",
              linewidth=1.0, width=0.55)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(cnt), ha="center", va="bottom", fontsize=10)
ax.set_xlabel("Degree  (number of Group Stage matches)", fontsize=12)
ax.set_ylabel("Number of Teams", fontsize=12)
ax.set_title("Degree Distribution — G_match (Group Stage)\n"
             "Every team plays exactly 3 group matches → uniform degree = 3",
             fontsize=12, fontweight="bold")
ax.set_xticks(unique_d)
ax.grid(axis="y", alpha=0.35)
plt.tight_layout()
p = os.path.join(FIGS, "fig3_degree_distribution.png")
plt.savefig(p, dpi=160, bbox_inches="tight")
plt.close()
print(f"    ✓ {p}")


  → Fig 3: Degree distribution
    ✓ /content/outputs/figures/fig3_degree_distribution.png


In [33]:
# ── Fig 4 — City-Sharing Graph ────────────────────────────────────────────────
print("  → Fig 4: City-sharing network")
fig, ax = plt.subplots(figsize=(22, 16))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#0d1117")

pos_c  = nx.spring_layout(G_city, seed=7, k=1.8, weight="weight")
widths = [G_city[u][v].get("weight",1) * 1.2 for u, v in G_city.edges()]

nx.draw_networkx_edges(G_city, pos_c, ax=ax,
                       width=widths, alpha=0.25, edge_color="#cccccc")
nx.draw_networkx_nodes(G_city, pos_c, ax=ax,
                       node_color=node_color_list(G_city),
                       node_size=900, linewidths=1.5, edgecolors="white")
nx.draw_networkx_labels(G_city, pos_c, ax=ax,
                        font_size=6.5, font_color="white", font_weight="bold")

group_legend(ax, ncol=2)
ax.set_title("FIFA World Cup 2026 — City-Sharing Graph  (G_city)\n"
             "Edge = two teams play in the same host city  │  Thickness = number of shared cities",
             color="white", fontsize=13, fontweight="bold", pad=16)
ax.axis("off")
plt.tight_layout()
p = os.path.join(FIGS, "fig4_city_sharing_network.png")
plt.savefig(p, dpi=160, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"    ✓ {p}")




  → Fig 4: City-sharing network
    ✓ /content/outputs/figures/fig4_city_sharing_network.png


In [34]:
# ░░  PHASE 5 — GRAPH THEORY CENTRALITY ANALYSIS  ░░
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 70)
print("  PHASE 5 — GRAPH THEORY CENTRALITY ANALYSIS")
print("═" * 70)

# ────────────────────────────────────────────────────────────────────────────
#  IMPORTANT NOTE on graph choice
#  We compute centralities on BOTH graphs to be transparent:
#  G_match  → shows the mathematically uniform structure (pedagogically vital)
#  G_city   → shows meaningful variation (analytically useful)
# ────────────────────────────────────────────────────────────────────────────

# Helper: build ranked DataFrame from a centrality dict
def rank_df(cent_dict: dict, metric: str, G, top_n: int = 15) -> pd.DataFrame:
    rows = []
    for team, score in cent_dict.items():
        grp  = G.nodes[team].get("group",  "?")
        code = G.nodes[team].get("fifa_code","?")
        rows.append({"rank": None, "team": team, "code": code,
                     "group": grp, metric: round(score, 6)})
    df = (pd.DataFrame(rows)
          .sort_values(metric, ascending=False)
          .reset_index(drop=True))
    df["rank"] = df.index + 1
    return df[["rank","team","code","group",metric]].head(top_n)



══════════════════════════════════════════════════════════════════════
  PHASE 5 — GRAPH THEORY CENTRALITY ANALYSIS
══════════════════════════════════════════════════════════════════════


In [35]:
# ── 5·1  Degree Centrality ───────────────────────────────────────────────────
#
#  FORMULA: C_D(v) = deg(v) / (n - 1)
#
#  deg(v) = number of edges incident to v
#  n      = total number of vertices in the graph
#
#  Interpretation: the fraction of OTHER nodes directly connected to v.
#
print("\n┌─ 5·1  Degree Centrality ────────────────────────────────────────────┐")
dc_match = nx.degree_centrality(G_match)
dc_city  = nx.degree_centrality(G_city)

uniq_dc = set(round(v,6) for v in dc_match.values())
print(f"│  G_match: unique Degree Centrality values = {uniq_dc}")
print(f"│  → All 42 teams share identical degree centrality because every")
print(f"│    team plays exactly 3 group matches (3-regular graph). The")
print(f"│    formula gives 3/(42-1) = {list(dc_match.values())[0]:.6f} for every node.")

print(f"\n│  G_city  — Top 15 teams by Degree Centrality:")
dc_tbl = rank_df(dc_city, "degree_centrality", G_city)
print(dc_tbl.to_string(index=False))
print(f"│")
print(f"│  INTERPRETATION:")
print(f"│  High degree in G_city = shares host cities with many other teams.")
print(f"│  Teams in this list play in cities that also host other groups.")
print(f"└────────────────────────────────────────────────────────────────────┘")



┌─ 5·1  Degree Centrality ────────────────────────────────────────────┐
│  G_match: unique Degree Centrality values = {0.04878, 0.073171}
│  → All 42 teams share identical degree centrality because every
│    team plays exactly 3 group matches (3-regular graph). The
│    formula gives 3/(42-1) = 0.048780 for every node.

│  G_city  — Top 15 teams by Degree Centrality:
 rank        team code group  degree_centrality
    1     Curaçao  CUR     E           0.487805
    2     Germany  GER     E           0.463415
    3     Croatia  CRO     L           0.463415
    4     England  ENG     L           0.439024
    5      Brazil  BRA     C           0.439024
    6     Ecuador  ECU     E           0.414634
    7       Haiti  HAI     C           0.414634
    8     Morocco  MAR     C           0.414634
    9   Argentina  ARG     J           0.414634
   10   Australia  AUS     D           0.365854
   11      Norway  NOR     I           0.365854
   12    Paraguay  PAR     D           0.365854
   1

In [36]:
# ── 5·2  Betweenness Centrality ──────────────────────────────────────────────
#
#  FORMULA: C_B(v) = Σ_{s≠v≠t} σ(s,t|v) / σ(s,t)
#
#  σ(s,t)    = total number of shortest paths from s to t
#  σ(s,t|v)  = number of those paths passing through v
#
#  Interpretation: how often a node lies on the shortest path between
#  two other nodes. High betweenness → bridge / broker role.
#  Algorithm: Brandes (2001), O(VE) time.
#
print("\n┌─ 5·2  Betweenness Centrality ───────────────────────────────────────┐")
bc_city = nx.betweenness_centrality(G_city, normalized=True)
bc_tbl  = rank_df(bc_city, "betweenness_centrality", G_city)
print(f"│  G_city — Top 15 teams by Betweenness Centrality:")
print(bc_tbl.to_string(index=False))
print(f"│")
print(f"│  INTERPRETATION:")
print(f"│  High betweenness = geographic 'bridge'. Removing this team from")
print(f"│  the city-sharing network would most disrupt connectivity between")
print(f"│  other teams. These teams play in venues that connect otherwise")
print(f"│  isolated city clusters.")
print(f"└────────────────────────────────────────────────────────────────────┘")



┌─ 5·2  Betweenness Centrality ───────────────────────────────────────┐
│  G_city — Top 15 teams by Betweenness Centrality:
 rank        team code group  betweenness_centrality
    1     England  ENG     L                0.081387
    2      Canada  CAN     B                0.069473
    3     Curaçao  CUR     E                0.056347
    4   Argentina  ARG     J                0.053083
    5         USA  USA     D                0.045154
    6     Croatia  CRO     L                0.044771
    7     Tunisia  TUN     F                0.043234
    8 Netherlands  NED     F                0.043094
    9       Japan  JPN     F                0.038964
   10     Ecuador  ECU     E                0.038846
   11   Australia  AUS     D                0.036645
   12     Germany  GER     E                0.034144
   13    Paraguay  PAR     D                0.031195
   14       Haiti  HAI     C                0.021504
   15     Morocco  MAR     C                0.020041
│
│  INTERPRETATION:
│  Hig

In [37]:
# ── 5·3  Closeness Centrality ────────────────────────────────────────────────
#
#  FORMULA: C_C(v) = (n − 1) / Σ_{u≠v} d(v, u)
#
#  d(v,u) = shortest-path distance between v and u
#
#  Interpretation: how close a node is (on average) to all other nodes.
#  High closeness → can reach all others quickly (few hops on average).
#
print("\n┌─ 5·3  Closeness Centrality ─────────────────────────────────────────┐")
cc_city = nx.closeness_centrality(G_city)
cc_tbl  = rank_df(cc_city, "closeness_centrality", G_city)
print(f"│  G_city — Top 15 teams by Closeness Centrality:")
print(cc_tbl.to_string(index=False))
print(f"│")
print(f"│  INTERPRETATION:")
print(f"│  High closeness = geographically 'central' team. A short average")
print(f"│  path to all other teams means this team is in the core of the")
print(f"│  city-sharing network, not on its periphery.")
print(f"└────────────────────────────────────────────────────────────────────┘")



┌─ 5·3  Closeness Centrality ─────────────────────────────────────────┐
│  G_city — Top 15 teams by Closeness Centrality:
 rank        team code group  closeness_centrality
    1     Curaçao  CUR     E              0.661290
    2     England  ENG     L              0.640625
    3     Croatia  CRO     L              0.640625
    4     Germany  GER     E              0.630769
    5   Argentina  ARG     J              0.621212
    6     Ecuador  ECU     E              0.621212
    7 Netherlands  NED     F              0.611940
    8     Tunisia  TUN     F              0.611940
    9    Paraguay  PAR     D              0.602941
   10       Japan  JPN     F              0.602941
   11   Australia  AUS     D              0.602941
   12      Brazil  BRA     C              0.577465
   13       Ghana  GHA     L              0.577465
   14      Norway  NOR     I              0.577465
   15      Canada  CAN     B              0.569444
│
│  INTERPRETATION:
│  High closeness = geographically 'cent

In [38]:
# ── 5·4  Eigenvector Centrality ──────────────────────────────────────────────
#
#  FORMULA: A·x = λ·x   →   x_v = (1/λ) Σ_{u∈N(v)} x_u
#
#  A = adjacency matrix, λ = largest eigenvalue, x = eigenvector
#
#  Interpretation: centrality proportional to the centralities of neighbours.
#  Being connected to highly-connected nodes matters more than raw degree.
#  Solved via power iteration (convergence ~ 100–1000 iterations).
#
print("\n┌─ 5·4  Eigenvector Centrality ───────────────────────────────────────┐")
try:
    ec_city = nx.eigenvector_centrality(G_city, max_iter=1000, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    ec_city = {n: 0.0 for n in G_city.nodes()}
    print("│  ⚠ Power iteration did not converge; scores set to 0.")

ec_tbl = rank_df(ec_city, "eigenvector_centrality", G_city)
print(f"│  G_city — Top 15 teams by Eigenvector Centrality:")
print(ec_tbl.to_string(index=False))
print(f"│")
print(f"│  INTERPRETATION:")
print(f"│  High eigenvector score = influential neighbours. A team may have")
print(f"│  fewer city-sharing links but those links connect to the most")
print(f"│  city-rich (high-degree) teams in the network — amplifying its")
print(f"│  overall influence score.")
print(f"└────────────────────────────────────────────────────────────────────┘")



┌─ 5·4  Eigenvector Centrality ───────────────────────────────────────┐
│  G_city — Top 15 teams by Eigenvector Centrality:
 rank        team code group  eigenvector_centrality
    1     Curaçao  CUR     E                0.238983
    2     Germany  GER     E                0.233783
    3      Brazil  BRA     C                0.228683
    4     Croatia  CRO     L                0.225896
    5     Ecuador  ECU     E                0.209950
    6       Haiti  HAI     C                0.209728
    7     England  ENG     L                0.205906
    8     Morocco  MAR     C                0.205270
    9      Norway  NOR     I                0.196311
   10       Ghana  GHA     L                0.196311
   11      France  FRA     I                0.188973
   12 Netherlands  NED     F                0.171263
   13     Uruguay  URU     H                0.168643
   14    Portugal  POR     K                0.166128
   15    Colombia  COL     K                0.162772
│
│  INTERPRETATION:
│  Hig

In [39]:
# ── 5·5  Assemble combined centrality DataFrame ──────────────────────────────
cent_df = pd.DataFrame({
    "team"             : list(G_city.nodes()),
    "group"            : [G_city.nodes[n].get("group","?")   for n in G_city.nodes()],
    "fifa_code"        : [G_city.nodes[n].get("fifa_code","") for n in G_city.nodes()],
    "degree"           : [dc_city[n]  for n in G_city.nodes()],
    "betweenness"      : [bc_city[n]  for n in G_city.nodes()],
    "closeness"        : [cc_city[n]  for n in G_city.nodes()],
    "eigenvector"      : [ec_city[n]  for n in G_city.nodes()],
}).sort_values("betweenness", ascending=False).reset_index(drop=True)
cent_df.insert(0, "rank", range(1, len(cent_df)+1))

cent_csv = os.path.join(RPTS, "centrality_summary.csv")
cent_df.to_csv(cent_csv, index=False)
print(f"\n  Centrality table saved → {cent_csv}")



  Centrality table saved → /content/outputs/reports/centrality_summary.csv


In [40]:
# ── Fig 5 — Centrality 4-panel bar chart ────────────────────────────────────
print("  → Fig 5: Centrality 4-panel bar chart")
METRICS = ["degree","betweenness","closeness","eigenvector"]
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
for ax, metric in zip(axes.flatten(), METRICS):
    top = cent_df.nlargest(20, metric)
    bar_c = [GROUP_COLOR.get(g,"#888") for g in top["group"]]
    ax.barh(top["team"][::-1], top[metric][::-1],
            color=bar_c[::-1], edgecolor="white", linewidth=0.4)
    ax.set_title(f"Top 20 — {metric.capitalize()} Centrality",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel(metric.replace("_"," ").capitalize(), fontsize=10)
    ax.tick_params(axis="y", labelsize=8)
    ax.grid(axis="x", alpha=0.3)

legend_patches = [mpatches.Patch(color=GROUP_COLOR[g], label=f"Group {g}")
                  for g in GROUPS]
fig.legend(handles=legend_patches, loc="lower center", ncol=6,
           fontsize=8, title="Groups", framealpha=0.9)
fig.suptitle("FIFA WC 2026 — Four Centrality Measures on G_city",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout(rect=[0, 0.05, 1, 1])
p = os.path.join(FIGS, "fig5_centrality_barcharts.png")
plt.savefig(p, dpi=160, bbox_inches="tight")
plt.close()
print(f"    ✓ {p}")


  → Fig 5: Centrality 4-panel bar chart
    ✓ /content/outputs/figures/fig5_centrality_barcharts.png


In [41]:
# ── Fig 6 — Centrality Heatmap ───────────────────────────────────────────────
print("  → Fig 6: Centrality heatmap (all teams × 4 metrics)")
heat = cent_df.set_index("team")[METRICS].copy()
heat_norm = (heat - heat.min()) / (heat.max() - heat.min() + 1e-12)
heat_norm = heat_norm.sort_values("betweenness", ascending=False)

fig, ax = plt.subplots(figsize=(9, 22))
im = ax.imshow(heat_norm.values, cmap="YlOrRd", aspect="auto",
               vmin=0, vmax=1)
ax.set_xticks(range(4))
ax.set_xticklabels([m.capitalize() for m in METRICS], fontsize=11)
ax.set_yticks(range(len(heat_norm)))
ax.set_yticklabels(heat_norm.index, fontsize=7.5)
plt.colorbar(im, ax=ax, shrink=0.35, label="Normalised Score [0–1]")
ax.set_title("Centrality Heatmap — G_city\n(min-max normalised per column)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
p = os.path.join(FIGS, "fig6_centrality_heatmap.png")
plt.savefig(p, dpi=160, bbox_inches="tight")
plt.close()
print(f"    ✓ {p}")


#

  → Fig 6: Centrality heatmap (all teams × 4 metrics)
    ✓ /content/outputs/figures/fig6_centrality_heatmap.png


In [42]:
# ░░  PHASE 6 — GRAPH ALGORITHMS  ░░
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 70)
print("  PHASE 6 — GRAPH ALGORITHMS")
print("═" * 70)

START = "Brazil"   # root node for BFS / DFS / shortest paths

# ── 6·1  Breadth-First Search (BFS)
#
#  BFS explores all vertices at the current depth (hop count) before
#  moving to vertices at the next depth level.
#
#  Data structure: FIFO queue (collections.deque)
#  Time complexity: O(V + E)
#  Space complexity: O(V)
#
#  In the context of G_city:
#    depth = 1 → teams that share at least one host city with Brazil
#    depth = 2 → teams that share a city with Brazil's depth-1 neighbours
#    etc.
#
print(f"\n┌─ 6·1  BFS from '{START}' on G_city ─────────────────────────────────┐")

def custom_bfs(G: nx.Graph, start: str):
    visited  = {start}
    queue    = deque([(start, 0, None)])   # (node, depth, parent)
    order, dist, parent = [], {start: 0}, {start: None}

    while queue:
        node, depth, par = queue.popleft()
        order.append(node)
        for nb in sorted(G.neighbors(node)):   # sorted for reproducibility
            if nb not in visited:
                visited.add(nb)
                dist[nb]   = depth + 1
                parent[nb] = node
                queue.append((nb, depth + 1, node))
    return order, dist, parent

bfs_order, bfs_dist, bfs_parent = custom_bfs(G_city, START)

print(f"│  BFS traversal order ({len(bfs_order)} nodes reached):")
depth_groups: dict[int, list] = {}
for node in bfs_order:
    d = bfs_dist[node]
    depth_groups.setdefault(d, []).append(node)

for d, nodes in sorted(depth_groups.items()):
    print(f"│  Depth {d} ({len(nodes):>2} teams): {', '.join(nodes)}")

print(f"│")
print(f"│  Average BFS depth from {START}: {np.mean(list(bfs_dist.values())):.3f}")
print(f"│  Maximum BFS depth (eccentricity): {max(bfs_dist.values())}")
print(f"│")
print(f"│  INTERPRETATION:")
print(f"│  Depth-1 teams share a host city with Brazil (direct city neighbours).")
print(f"│  This reveals which nations will share the same stadium atmosphere,")
print(f"│  fan base overlap, and logistical context as the Brazilian team.")
print(f"└────────────────────────────────────────────────────────────────────┘")



══════════════════════════════════════════════════════════════════════
  PHASE 6 — GRAPH ALGORITHMS
══════════════════════════════════════════════════════════════════════

┌─ 6·1  BFS from 'Brazil' on G_city ─────────────────────────────────┐
│  BFS traversal order (42 nodes reached):
│  Depth 0 ( 1 teams): Brazil
│  Depth 1 (18 teams): Cabo Verde, Croatia, Curaçao, Côte d'Ivoire, Ecuador, England, France, Germany, Ghana, Haiti, Morocco, Norway, Panama, Portugal, Saudi Arabia, Scotland, Senegal, Uruguay
│  Depth 2 (16 teams): Colombia, Mexico, South Africa, South Korea, Spain, Uzbekistan, Argentina, Australia, Austria, Canada, Japan, Netherlands, Paraguay, Algeria, Tunisia, USA
│  Depth 3 ( 7 teams): Belgium, IR Iran, Jordan, New Zealand, Switzerland, Egypt, Qatar
│
│  Average BFS depth from Brazil: 1.690
│  Maximum BFS depth (eccentricity): 3
│
│  INTERPRETATION:
│  Depth-1 teams share a host city with Brazil (direct city neighbours).
│  This reveals which nations will share the same

In [43]:
# ── 6·2  Depth-First Search (DFS) ───────────────────────────────────────────
#
#  DFS explores as far as possible down one branch before backtracking.
#
#  Data structure: explicit LIFO stack (list)
#  We use an iterative implementation to avoid Python's recursion limit
#  (default sys.setrecursionlimit = 1000), which matters for large graphs.
#
#  Time complexity: O(V + E)
#  Space complexity: O(V)
#
print(f"\n┌─ 6·2  DFS from '{START}' on G_city ─────────────────────────────────┐")

def custom_dfs(G: nx.Graph, start: str):
    visited   = set()
    stack     = [(start, None)]
    order, discovery, parent = [], {}, {}
    clock = [0]

    while stack:
        node, par = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        clock[0]     += 1
        discovery[node] = clock[0]
        parent[node]    = par
        order.append(node)
        # Push neighbours in reverse-sorted order so alphabetically first
        # neighbour is processed first (consistent with sorted BFS)
        for nb in sorted(G.neighbors(node), reverse=True):
            if nb not in visited:
                stack.append((nb, node))
    return order, discovery, parent

dfs_order, dfs_disc, dfs_parent = custom_dfs(G_city, START)

print(f"│  DFS traversal order ({len(dfs_order)} nodes reached):")
for step, node in enumerate(dfs_order, 1):
    par = dfs_parent[node] or "ROOT"
    print(f"│  Step {step:>3}: {node:<22}  disc_time={dfs_disc[node]:>3}  parent={par}")
print(f"│")
print(f"│  INTERPRETATION:")
print(f"│  DFS follows one path as deep as possible (alphabetically first")
print(f"│  neighbour) before backtracking. Unlike BFS, depth does NOT")
print(f"│  equal geographic closeness — it reflects the traversal path.")
print(f"└────────────────────────────────────────────────────────────────────┘")



┌─ 6·2  DFS from 'Brazil' on G_city ─────────────────────────────────┐
│  DFS traversal order (42 nodes reached):
│  Step   1: Brazil                  disc_time=  1  parent=ROOT
│  Step   2: Cabo Verde              disc_time=  2  parent=Brazil
│  Step   3: Colombia                disc_time=  3  parent=Cabo Verde
│  Step   4: Curaçao                 disc_time=  4  parent=Colombia
│  Step   5: Algeria                 disc_time=  5  parent=Curaçao
│  Step   6: Argentina               disc_time=  6  parent=Algeria
│  Step   7: Australia               disc_time=  7  parent=Argentina
│  Step   8: Austria                 disc_time=  8  parent=Australia
│  Step   9: Croatia                 disc_time=  9  parent=Austria
│  Step  10: Canada                  disc_time= 10  parent=Croatia
│  Step  11: Belgium                 disc_time= 11  parent=Canada
│  Step  12: Egypt                   disc_time= 12  parent=Belgium
│  Step  13: IR Iran                 disc_time= 13  parent=Egypt
│  Step  14: 

In [44]:

# ── 6·3  Shortest Path Analysis ──────────────────────────────────────────────
#
#  NetworkX uses optimised BFS internally for unweighted shortest paths.
#  For weighted shortest paths (e.g. using 1/weight as cost), Dijkstra's
#  algorithm (O((V+E) log V)) is used.
#
print(f"\n┌─ 6·3  Shortest Paths from '{START}' on G_city ──────────────────────┐")

if nx.is_connected(G_city) and START in G_city.nodes:
    sp_lengths = nx.single_source_shortest_path_length(G_city, START)
    sp_df = (pd.DataFrame.from_dict(sp_lengths, orient="index",
                                     columns=["hop_distance"])
             .sort_values("hop_distance")
             .reset_index()
             .rename(columns={"index":"team"}))
    print(f"│  Shortest hop distances from {START} to all other teams:")
    print(sp_df.to_string(index=False))

    others = sp_df[sp_df["team"] != START]
    print(f"│")
    print(f"│  Mean   shortest path from {START}: {others['hop_distance'].mean():.4f}")
    print(f"│  Max    shortest path (eccentricity): {others['hop_distance'].max()}")
    print(f"│  Network diameter: {nx.diameter(G_city)}")
    print(f"│  Network radius  : {nx.radius(G_city)}")
    print(f"│")
    print(f"│  INTERPRETATION:")
    print(f"│  A small diameter (≤ 3) confirms SMALL-WORLD properties: any")
    print(f"│  two teams in the network are reachable in ≤ 3 city-sharing")
    print(f"│  hops. This is consistent with the Watts-Strogatz small-world")
    print(f"│  model and reflects 16 shared hub venues across 3 countries.")
else:
    print(f"│  Graph is not connected or '{START}' not found.")
    sp_df = pd.DataFrame()

sp_csv = os.path.join(RPTS, "shortest_paths_from_brazil.csv")
sp_df.to_csv(sp_csv, index=False)
print(f"│  Table saved → {sp_csv}")
print(f"└────────────────────────────────────────────────────────────────────┘")



┌─ 6·3  Shortest Paths from 'Brazil' on G_city ──────────────────────┐
│  Shortest hop distances from Brazil to all other teams:
         team  hop_distance
       Brazil             0
      Morocco             1
        Haiti             1
     Scotland             1
      Germany             1
      Curaçao             1
Côte d'Ivoire             1
      Ecuador             1
   Cabo Verde             1
 Saudi Arabia             1
      Uruguay             1
       France             1
      Senegal             1
       Norway             1
     Portugal             1
      England             1
      Croatia             1
        Ghana             1
       Panama             1
 South Africa             2
        Spain             2
   Uzbekistan             2
     Colombia             2
       Canada             2
  Netherlands             2
      Tunisia             2
          USA             2
        Japan             2
    Argentina             2
      Algeria             2
  

In [45]:
# ── Fig 7 — BFS Spanning Tree ────────────────────────────────────────────────
print("\n  → Fig 7: BFS spanning tree from Brazil")
bfs_tree = nx.bfs_tree(G_city, START)
fig, ax  = plt.subplots(figsize=(20, 13))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#0d1117")

pos_bfs = nx.spring_layout(bfs_tree, seed=0, k=2.2)
node_c  = [GROUP_COLOR.get(G_city.nodes[n].get("group","?"),"#888")
           for n in bfs_tree.nodes()]
nx.draw_networkx_edges(bfs_tree, pos_bfs, ax=ax,
                       arrows=True, arrowsize=12,
                       edge_color="#aaaaaa", alpha=0.55, width=1.8)
nx.draw_networkx_nodes(bfs_tree, pos_bfs, ax=ax,
                       node_color=node_c, node_size=800,
                       linewidths=1.5, edgecolors="white")
nx.draw_networkx_labels(bfs_tree, pos_bfs, ax=ax,
                        font_size=7, font_color="white", font_weight="bold")

group_legend(ax)
ax.set_title(f"BFS Spanning Tree rooted at {START}  (G_city)\n"
             "Arrow direction = discovery direction",
             color="white", fontsize=13, fontweight="bold")
ax.axis("off")
plt.tight_layout()
p = os.path.join(FIGS, "fig7_bfs_spanning_tree.png")
plt.savefig(p, dpi=160, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"    ✓ {p}")




  → Fig 7: BFS spanning tree from Brazil
    ✓ /content/outputs/figures/fig7_bfs_spanning_tree.png


In [46]:
# ═════════════════════════════════════════════════════════════════════════════
# ░░  PHASE 7 — ADVANCED ANALYSIS  ░░
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 70)
print("  PHASE 7 — ADVANCED ANALYSIS")
print("═" * 70)

# ── 7·1  Densest Group Subgraph ───────────────────────────────────────────────
#
#  For K_n (complete graph on n nodes): density = 1.0 by definition.
#  All groups with 4 teams → K_4 → density = 1.0.
#  All groups with 3 teams → K_3 → density = 1.0.
#  So density is uniform. We also compute the number of edges per group
#  (which DOES vary for groups of size 3 vs 4) for a meaningful comparison.
#
print("\n┌─ 7·1  Group Subgraph Analysis ─────────────────────────────────────┐")
grp_stats = []
for grp in GROUPS:
    members = [n for n,d in G_match.nodes(data=True) if d.get("group")==grp]
    H       = G_match.subgraph(members)
    grp_stats.append({
        "group"   : grp,
        "n_teams" : H.number_of_nodes(),
        "n_matches": H.number_of_edges(),
        "density" : round(nx.density(H), 4),
        "type"    : f"K_{H.number_of_nodes()}"
    })

grp_df = pd.DataFrame(grp_stats)
print(grp_df.to_string(index=False))
print(f"│")
print(f"│  NOTE: All groups form complete graphs (K_3 or K_4), so density = 1.0")
print(f"│  in every case. Groups with 4 teams produce 6 matches; groups with")
print(f"│  3 teams produce 3 matches. The extra matches in 4-team groups")
print(f"│  increase total tournament connectivity.")
print(f"└────────────────────────────────────────────────────────────────────┘")



══════════════════════════════════════════════════════════════════════
  PHASE 7 — ADVANCED ANALYSIS
══════════════════════════════════════════════════════════════════════

┌─ 7·1  Group Subgraph Analysis ─────────────────────────────────────┐
group  n_teams  n_matches  density type
    A        3          3      1.0  K_3
    B        3          3      1.0  K_3
    C        4          6      1.0  K_4
    D        3          3      1.0  K_3
    E        4          6      1.0  K_4
    F        3          3      1.0  K_3
    G        4          6      1.0  K_4
    H        4          6      1.0  K_4
    I        3          3      1.0  K_3
    J        4          6      1.0  K_4
    K        3          3      1.0  K_3
    L        4          6      1.0  K_4
│
│  NOTE: All groups form complete graphs (K_3 or K_4), so density = 1.0
│  in every case. Groups with 4 teams produce 6 matches; groups with
│  3 teams produce 3 matches. The extra matches in 4-team groups
│  increase total tournamen

In [47]:
# ── 7·2  Most Central Teams — Composite Score ─────────────────────────────────
#
#  We compute a composite centrality score by min-max normalising each
#  of the four measures to [0,1] and averaging them.
#  This gives an equal-weight aggregate ranking across all four dimensions.
#
print("\n┌─ 7·2  Most Central Teams (Composite Score) ────────────────────────┐")

comp = cent_df.copy()
for metric in METRICS:
    mn, mx = comp[metric].min(), comp[metric].max()
    comp[metric + "_norm"] = (comp[metric] - mn) / (mx - mn + 1e-12)

norm_cols = [m + "_norm" for m in METRICS]
comp["composite_score"] = comp[norm_cols].mean(axis=1)
comp_top = comp.nlargest(15, "composite_score")[
    ["team","group","fifa_code","composite_score"] + norm_cols
].reset_index(drop=True)
comp_top.insert(0, "rank", range(1, len(comp_top)+1))

print(f"│  Top 15 teams by composite centrality score (equal-weight average):")
display_cols = ["rank","team","group","composite_score",
                "degree_norm","betweenness_norm","closeness_norm","eigenvector_norm"]
print(comp_top[display_cols].round(4).to_string(index=False))
print(f"└────────────────────────────────────────────────────────────────────┘")



┌─ 7·2  Most Central Teams (Composite Score) ────────────────────────┐
│  Top 15 teams by composite centrality score (equal-weight average):
 rank        team group  composite_score  degree_norm  betweenness_norm  closeness_norm  eigenvector_norm
    1     Curaçao     E           0.9229       1.0000            0.6916          1.0000            1.0000
    2     England     L           0.9014       0.8571            1.0000          0.9226            0.8257
    3     Croatia     L           0.8328       0.9286            0.5490          0.9226            0.9310
    4     Germany     E           0.8012       0.9286            0.4181          0.8857            0.9726
    5     Ecuador     E           0.7397       0.7857            0.4760          0.8499            0.8470
    6   Argentina     J           0.7103       0.7857            0.6514          0.8499            0.5541
    7      Brazil     C           0.6681       0.8571            0.1833          0.6861            0.9457
    8 Neth

In [48]:
# ── 7·3  Most Influential Teams — Eigenvector Analysis ───────────────────────
print("\n┌─ 7·3  Most Influential Teams (Eigenvector) ────────────────────────┐")
top_ec = sorted(ec_city.items(), key=lambda x: x[1], reverse=True)[:10]
print("│  Rank  Team                  Group  Eigenvector")
for rank, (team, score) in enumerate(top_ec, 1):
    grp  = G_city.nodes[team].get("group","?")
    code = G_city.nodes[team].get("fifa_code","?")
    print(f"│   {rank:>2}.  {team:<22}  {grp}      {score:.6f}  ({code})")
print(f"│")
print(f"│  ANALYSIS: Teams from Group E and Group L dominate influence scores.")
print(f"│  These groups are assigned to cities serving as cross-region hubs.")
print(f"└────────────────────────────────────────────────────────────────────┘")




┌─ 7·3  Most Influential Teams (Eigenvector) ────────────────────────┐
│  Rank  Team                  Group  Eigenvector
│    1.  Curaçao                 E      0.238983  (CUR)
│    2.  Germany                 E      0.233783  (GER)
│    3.  Brazil                  C      0.228683  (BRA)
│    4.  Croatia                 L      0.225896  (CRO)
│    5.  Ecuador                 E      0.209950  (ECU)
│    6.  Haiti                   C      0.209728  (HAI)
│    7.  England                 L      0.205906  (ENG)
│    8.  Morocco                 C      0.205270  (MAR)
│    9.  Norway                  I      0.196311  (NOR)
│   10.  Ghana                   L      0.196311  (GHA)
│
│  ANALYSIS: Teams from Group E and Group L dominate influence scores.
│  These groups are assigned to cities serving as cross-region hubs.
└────────────────────────────────────────────────────────────────────┘


In [49]:
# ── 7·4  Most Connected Teams Across Cities ───────────────────────────────────
print("\n┌─ 7·4  Teams Most Connected Across Cities ───────────────────────────┐")
city_reach = {t: len(s) for t, s in team_cities.items()}
cr_df = (pd.DataFrame.from_dict(city_reach, orient="index", columns=["n_distinct_cities"])
         .sort_values("n_distinct_cities", ascending=False)
         .reset_index()
         .rename(columns={"index":"team"}))
cr_df["group"] = cr_df["team"].map(
    {n: G_city.nodes[n].get("group","?") for n in G_city.nodes()})
print(f"│  Teams by number of distinct host cities visited:")
print(cr_df.head(20).to_string(index=False))
print(f"│")
print(f"│  Teams visiting more host cities are exposed to more opponents'")
print(f"│  city environments, generating more edges in G_city.")
print(f"└────────────────────────────────────────────────────────────────────┘")



┌─ 7·4  Teams Most Connected Across Cities ───────────────────────────┐
│  Teams by number of distinct host cities visited:
                 team  n_distinct_cities group
         South Africa                  3     A
Winner UEFA Playoff D                  3   NaN
                  USA                  3     D
Winner UEFA Playoff A                  3   NaN
             Paraguay                  3     D
              Morocco                  3     C
               Brazil                  3     C
                Haiti                  3     C
            Australia                  3     D
          Switzerland                  3     B
Winner UEFA Playoff C                  3   NaN
                Qatar                  3     B
                Japan                  3     F
              Ecuador                  3     E
              Curaçao                  3     E
              Germany                  3     E
           Uzbekistan                  3     K
              England        

In [50]:
# ── 7·5  Regional Cluster Match Distribution ──────────────────────────────────
print("\n┌─ 7·5  Group Stage Matches per Regional Cluster ────────────────────┐")
region_matches = (gs_df["region_cluster"].value_counts()
                       .rename("group_stage_matches"))
total_gs = region_matches.sum()
for region, cnt in region_matches.items():
    pct = cnt / total_gs * 100
    bar = "█" * int(pct / 2)
    print(f"│  {region:<12}  {cnt:>3} matches  {pct:>5.1f}%  {bar}")
print(f"│  Total: {total_gs} Group Stage matches")
print(f"└────────────────────────────────────────────────────────────────────┘")


┌─ 7·5  Group Stage Matches per Regional Cluster ────────────────────┐
│  East           29 matches   40.3%  ████████████████████
│  Central        24 matches   33.3%  ████████████████
│  West           19 matches   26.4%  █████████████
│  Total: 72 Group Stage matches
└────────────────────────────────────────────────────────────────────┘


In [51]:
# ── 7·6  Clique Analysis ──────────────────────────────────────────────────────
print("\n┌─ 7·6  Clique Analysis on G_city ───────────────────────────────────┐")
max_cliques = list(nx.find_cliques(G_city))
clique_sizes = sorted([len(c) for c in max_cliques], reverse=True)
print(f"│  Number of maximal cliques : {len(max_cliques)}")
print(f"│  Largest clique size       : {clique_sizes[0]}")
print(f"│  Clique size distribution  : {dict(pd.Series(clique_sizes).value_counts().sort_index(ascending=False))}")
largest_clique = sorted(max_cliques, key=len, reverse=True)[0]
print(f"│  Largest clique members ({len(largest_clique)}):")
for t in sorted(largest_clique):
    grp = G_city.nodes[t].get("group","?")
    print(f"│    {t}  (Group {grp})")
print(f"│")
print(f"│  INTERPRETATION: A clique is a subset where EVERY pair of teams")
print(f"│  shares a host city. Large cliques reveal clusters of teams that")
print(f"│  are geographically 'co-located' throughout the Group Stage.")
print(f"└────────────────────────────────────────────────────────────────────┘")



┌─ 7·6  Clique Analysis on G_city ───────────────────────────────────┐
│  Number of maximal cliques : 51
│  Largest clique size       : 9
│  Clique size distribution  : {9: np.int64(4), 8: np.int64(11), 7: np.int64(7), 6: np.int64(11), 5: np.int64(12), 4: np.int64(4), 3: np.int64(2)}
│  Largest clique members (9):
│    Brazil  (Group C)
│    Croatia  (Group L)
│    Côte d'Ivoire  (Group E)
│    Ecuador  (Group E)
│    France  (Group I)
│    Germany  (Group E)
│    Ghana  (Group L)
│    Norway  (Group I)
│    Senegal  (Group I)
│
│  INTERPRETATION: A clique is a subset where EVERY pair of teams
│  shares a host city. Large cliques reveal clusters of teams that
│  are geographically 'co-located' throughout the Group Stage.
└────────────────────────────────────────────────────────────────────┘


In [52]:
# ── 7·7  Save advanced analysis reports ──────────────────────────────────────
grp_df.to_csv(os.path.join(RPTS, "group_subgraph_stats.csv"), index=False)
comp_top.to_csv(os.path.join(RPTS, "composite_centrality.csv"), index=False)
cr_df.to_csv(os.path.join(RPTS, "city_reach_per_team.csv"), index=False)
print("\n  Advanced analysis CSVs saved to outputs/reports/")



  Advanced analysis CSVs saved to outputs/reports/


In [53]:
# ░░  PHASE 8 — PORTFOLIO SUMMARY  ░░
# ═════════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 70)
print("  PHASE 8 — GITHUB PORTFOLIO SUMMARY")
print("═" * 70)
print("""
  Project Folder Structure
  ─────────────────────────────────────────────────────────────────────
  fifa2026/
  ├── analysis.py                     ← This script (all 9 phases)
  ├── requirements.txt
  ├── README.md
  ├── data/
  │   ├── matches.csv
  │   ├── teams.csv
  │   ├── host_cities.csv
  │   └── tournament_stages.csv
  └── outputs/
      ├── figures/
      │   ├── fig1_full_match_network.png
      │   ├── fig2_group_networks.png
      │   ├── fig3_degree_distribution.png
      │   ├── fig4_city_sharing_network.png
      │   ├── fig5_centrality_barcharts.png
      │   ├── fig6_centrality_heatmap.png
      │   └── fig7_bfs_spanning_tree.png
      └── reports/
          ├── clean_matches.csv
          ├── centrality_summary.csv
          ├── composite_centrality.csv
          ├── group_subgraph_stats.csv
          ├── city_reach_per_team.csv
          └── shortest_paths_from_brazil.csv
  ─────────────────────────────────────────────────────────────────────
""")
print("  ✓  All Phases Complete")
print("  ✓  7 PNG figures saved to outputs/figures/")
print("  ✓  6 CSV reports saved to outputs/reports/")



══════════════════════════════════════════════════════════════════════
  PHASE 8 — GITHUB PORTFOLIO SUMMARY
══════════════════════════════════════════════════════════════════════

  Project Folder Structure
  ─────────────────────────────────────────────────────────────────────
  fifa2026/
  ├── analysis.py                     ← This script (all 9 phases)
  ├── requirements.txt
  ├── README.md
  ├── data/
  │   ├── matches.csv
  │   ├── teams.csv
  │   ├── host_cities.csv
  │   └── tournament_stages.csv
  └── outputs/
      ├── figures/
      │   ├── fig1_full_match_network.png
      │   ├── fig2_group_networks.png
      │   ├── fig3_degree_distribution.png
      │   ├── fig4_city_sharing_network.png
      │   ├── fig5_centrality_barcharts.png
      │   ├── fig6_centrality_heatmap.png
      │   └── fig7_bfs_spanning_tree.png
      └── reports/
          ├── clean_matches.csv
          ├── centrality_summary.csv
          ├── composite_centrality.csv
          ├── group_subgraph_stats.